In [1]:
import pandas as pd

raw = '../data/raw/'
date_cols = ['order_purchase_timestamp', 'order_approved_at',
             'order_delivered_carrier_date',
             'order_delivered_customer_date',
             'order_estimated_delivery_date']

orders    = pd.read_csv(raw + 'olist_orders_dataset.csv', parse_dates=date_cols)
items     = pd.read_csv(raw + 'olist_order_items_dataset.csv')
customers = pd.read_csv(raw + 'olist_customers_dataset.csv')
products  = pd.read_csv(raw + 'olist_products_dataset.csv')
reviews   = pd.read_csv(raw + 'olist_order_reviews_dataset.csv')
cat_en    = pd.read_csv(raw + 'product_category_name_translation.csv')

print('Orders at start:', len(orders))

Orders at start: 99441


In [2]:
orders = orders[orders['order_status'] == 'delivered'].copy()
print('After keeping delivered:', len(orders))

orders = orders.dropna(subset=['order_delivered_customer_date'])
print('After removing missing delivery date:', len(orders))

After keeping delivered: 96478
After removing missing delivery date: 96470


In [3]:
orders = orders[(orders['order_purchase_timestamp'] >= '2017-01-01') &
                (orders['order_purchase_timestamp'] <  '2018-09-01')]
print('After date range filter:', len(orders))

After date range filter: 96203


In [4]:
reviews = (reviews.sort_values('review_creation_date')
                  .drop_duplicates('order_id', keep='last'))
print('Reviews after removing duplicates:', len(reviews))
print('Duplicates left:', reviews['order_id'].duplicated().sum())

Reviews after removing duplicates: 98673
Duplicates left: 0


In [5]:
products['product_category_name'] = products['product_category_name'].fillna('unknown')
products = products.merge(cat_en, on='product_category_name', how='left')
products['category_en'] = products['product_category_name_english'].fillna(
    products['product_category_name'])

print(products['category_en'].isna().sum())
products['category_en'].value_counts().head(10)

0


category_en
bed_bath_table           3029
sports_leisure           2867
furniture_decor          2657
health_beauty            2444
housewares               2335
auto                     1900
computers_accessories    1639
toys                     1411
watches_gifts            1329
telephony                1134
Name: count, dtype: int64

In [6]:
orders['delivery_days'] = (orders['order_delivered_customer_date']
                           - orders['order_purchase_timestamp']).dt.days

orders['is_late'] = (orders['order_delivered_customer_date']
                     > orders['order_estimated_delivery_date']).astype(int)

orders['order_month'] = orders['order_purchase_timestamp'].dt.to_period('M').astype(str)

orders[['delivery_days', 'is_late', 'order_month']].head()

,delivery_days,is_late,order_month
0,8,0,2017-10
1,13,0,2018-07
2,9,0,2018-08
3,13,0,2017-11
4,2,0,2018-02


In [7]:
print('order_id unique:', orders['order_id'].is_unique)
print('Negative delivery days:', (orders['delivery_days'] < 0).sum())
print('Late delivery rate:', round(orders['is_late'].mean() * 100, 1), '%')
orders['delivery_days'].describe()

order_id unique: True
Negative delivery days: 0
Late delivery rate: 8.1 %


count    96203.000000
mean        12.073854
std          9.535542
min          0.000000
25%          6.000000
50%         10.000000
75%         15.000000
max        209.000000
Name: delivery_days, dtype: float64

In [8]:
orders.to_csv('../data/clean/orders_clean.csv', index=False)
reviews.to_csv('../data/clean/reviews_clean.csv', index=False)
products[['product_id', 'category_en']].to_csv('../data/clean/products_clean.csv', index=False)
print('Saved')

Saved
